<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula07a_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip titanic.zip

Archive:  titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [2]:
import pandas as pd
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [4]:
y = df['Survived']
X = df.drop('Survived', axis=1)

In [5]:
len(y)

891

In [6]:
for col in X.columns:
  print(f"{col:>12} {len(set(X[col])):4} {X[col].dtype}")

 PassengerId  891 int64
      Pclass    3 int64
        Name  891 object
         Sex    2 object
         Age  265 float64
       SibSp    7 int64
       Parch    7 int64
      Ticket  681 object
        Fare  248 float64
       Cabin  148 object
    Embarked    4 object


In [7]:
caracteristicas_indesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
Xdrop = X.drop(caracteristicas_indesejadas, axis=1)
Xdrop.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [8]:
Xnum = Xdrop.select_dtypes('number')
Xnum.columns

Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')

In [9]:
for col in Xnum.columns:
  print(f"{col:>12} {Xnum[col].isnull().sum():2}")

      Pclass  0
         Age 177
       SibSp  0
       Parch  0
        Fare  0


In [11]:
Xnum[-5:]

,Pclass,Age,SibSp,Parch,Fare
886,2,27.0,0,0,13.00
887,1,19.0,0,0,30.00
888,3,NaN,1,2,23.45
889,1,26.0,0,0,30.00
890,3,32.0,0,0,7.75


In [13]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
XnumTratado = imputer.fit_transform(Xnum)
XnumTratado[-5:]

array([[ 2.  , 27.  ,  0.  ,  0.  , 13.  ],
       [ 1.  , 19.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 28.  ,  1.  ,  2.  , 23.45],
       [ 1.  , 26.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 32.  ,  0.  ,  0.  ,  7.75]])

In [14]:
Xcat = Xdrop.select_dtypes(object)
Xcat.columns

Index(['Sex', 'Embarked'], dtype='object')

In [15]:
for col in Xcat.columns:
  print(f"{col:>12} {Xcat[col].isnull().sum():2}")

         Sex  0
    Embarked  2


In [16]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
XcatTratado = imputer.fit_transform(Xcat)
XcatTratado[-5:]

array([['male', 'S'],
       ['female', 'S'],
       ['female', 'S'],
       ['male', 'C'],
       ['male', 'Q']], dtype=object)

In [17]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
XcatTratadoHot = encoder.fit_transform(XcatTratado)
XcatTratadoHot

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1782 stored elements and shape (891, 5)>

In [18]:
XcatTratadoHot.toarray()[-5:]

array([[0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [20]:
import numpy as np
Xtratado = np.c_[XnumTratado, XcatTratadoHot.toarray()]
Xtratado.shape

(891, 10)

In [26]:
from sklearn. base import BaseEstimator, TransformerMixin

class AtributosDesejados(BaseEstimator, TransformerMixin):
  def __init__(self):
    self.colunas_indesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.drop(self.colunas_indesejadas, axis=1)

atributos_desejados = AtributosDesejados()
Xdrop = atributos_desejados.fit_transform(X)
Xdrop.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [28]:
class AtributosNumericos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.select_dtypes(include='number')

atributos_numericos = AtributosNumericos()
Xnum = atributos_numericos.fit_transform(Xdrop)
Xnum[-5:]

,Pclass,Age,SibSp,Parch,Fare
886,2,27.0,0,0,13.00
887,1,19.0,0,0,30.00
888,3,NaN,1,2,23.45
889,1,26.0,0,0,30.00
890,3,32.0,0,0,7.75


In [29]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipenum = Pipeline([
    ('atributos_numericos', AtributosNumericos()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

XnumTratado = pipenum.fit_transform(Xdrop)
XnumTratado[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783]])

In [30]:
class AtributosCategoricos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    return self
  def transform(self, X, y=None):
    return X.select_dtypes(include='object')

atributos_categoricos = AtributosCategoricos()
Xcat = atributos_categoricos.fit_transform(Xdrop)
Xcat[-5:]

,Sex,Embarked
886,male,S
887,female,S
888,female,S
889,male,C
890,male,Q


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pipecat = Pipeline([
    ('atributos_categoricos', AtributosCategoricos()),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])

XcatTratado = pipecat.fit_transform(Xdrop)
XcatTratado

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1782 stored elements and shape (891, 5)>

In [36]:
from sklearn.pipeline import FeatureUnion

unecaracteristicas = FeatureUnion([
    ('pipenum', pipenum),
    ('pipecat', pipecat)
])

Xtratado = unecaracteristicas.fit_transform(Xdrop)
Xtratado.shape

(891, 10)

In [46]:
class DenseTransformer(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None, **fit_params):
    return self
  def transform(self, X, y=None, **fit_params):
    return X.toarray()

preproc = Pipeline([
    ('atributos_desejados', AtributosDesejados()),
    ('unecaracteristicas', unecaracteristicas),
    ('to_dense', DenseTransformer())
])

XcatTratado = preproc.fit_transform(X)
XcatTratado.shape

(891, 10)

In [55]:
from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([
    ('preproc', preproc),
    ('classificador', RandomForestClassifier())
])

clf.fit(X, y)

Pipeline(steps=[('preproc',
                 Pipeline(steps=[('atributos_desejados', AtributosDesejados()),
                                 ('unecaracteristicas',
                                  FeatureUnion(transformer_list=[('pipenum',
                                                                  Pipeline(steps=[('atributos_numericos',
                                                                                   AtributosNumericos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='median')),
                                                                                  ('scaler',
                                                                                   StandardScaler())])),
                                                                 ('pipecat',
                                                                  Pipeline(steps=[('atributos_categoricos',
                                                                                   AtributosCategoricos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                  ('encoder',
                                                                                   OneHotEncoder())]))])),
                                 ('to_dense', DenseTransformer())])),
                ('classificador', RandomForestClassifier())])

In [56]:
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

y_pred = clf.predict(X)
accuracy_score(y, y_pred)

0.9797979797979798

In [57]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(clf, X, y)
print(f"{scores.mean()}\n{scores}")

0.8070052099679869
[0.76536313 0.81460674 0.85393258 0.7752809  0.8258427 ]


In [58]:
clf.fit(X, y)
Xtest = pd.read_csv('test.csv')
ypred = clf.predict(Xtest)
ypred

array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [59]:
submission = pd.read_csv('gender_submission.csv')
submission['Survived'] = ypred
submission.to_csv('submission.csv', index=False)

In [60]:
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier

base_estimators = [
    ('rf', RandomForestClassifier()),
    ('et', ExtraTreesClassifier()),
    ('mlp', MLPClassifier()),
    ('sgd', SGDClassifier()),
    ('ridge', RidgeClassifier()),
    ('knn', KNeighborsClassifier()),
    ('gnb', GaussianNB()),
    ('lr', LogisticRegression())
]

clf = StackingClassifier(
    estimators=base_estimators
)

model = Pipeline([
    ('preproc', preproc),
    ('clf', clf)
])

scores = cross_val_score(model, X, y, cv=StratifiedKFold(n_splits=5))
print(f"{scores.mean()}\n{scores}")

0.8238026489234826
[0.81564246 0.81460674 0.8258427  0.80337079 0.85955056]


In [62]:
model.fit(X, y)
y_stk_pred = model.predict(Xtest)
y_stk_pred

array([0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [65]:
len(ypred)

418

In [66]:
sum(ypred != y_stk_pred)

np.int64(38)

In [67]:
submission = pd.read_csv('gender_submission.csv')
submission['Survived'] = y_stk_pred
submission.to_csv('submission_stk.csv', index=False)

In [70]:
model = Pipeline([
    ('preproc', preproc),
    ('classificador', MLPClassifier())
])

scores = cross_val_score(model, X, y, cv=StratifiedKFold(n_splits=5))
print(f"{scores.mean()}\n{scores}")

0.8215617349821104
[0.81005587 0.80898876 0.82022472 0.78651685 0.88202247]


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier

params = [
    {
        'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
        'activation': ['tanh', 'relu'],
        'solver': ['sgd', 'adam'],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate_init': [0.001, 0.01]
    }
]

clf = GridSearchCV(MLPClassifier(max_iter=1000), param_grid=params, cv=5, verbose=2, n_jobs=-1)

model = Pipeline([
    ('preproc', preproc),
    ('clf', clf)
])

scores = cross_val_score(model, X, y, cv=StratifiedKFold(n_splits=5), verbose=2, n_jobs=-1)
print(f"{scores.mean()}\n{scores}")

In [72]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.2 MB/s eta 0:00:00


In [73]:
import optuna
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

def objective(trial):
    # Hyperparameters to tune for MLPClassifier
    hidden_layer_sizes_choice = trial.suggest_categorical('hidden_layer_sizes', [(50,), (100,), (50, 50), (100, 50, 25)])
    activation = trial.suggest_categorical('activation', ['tanh', 'relu'])
    solver = trial.suggest_categorical('solver', ['sgd', 'adam'])
    alpha = trial.suggest_loguniform('alpha', 1e-4, 1e-2)
    learning_rate_init = trial.suggest_loguniform('learning_rate_init', 1e-3, 1e-1)

    clf = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes_choice,
        activation=activation,
        solver=solver,
        alpha=alpha,
        learning_rate_init=learning_rate_init,
        max_iter=1000, # Increased max_iter to ensure convergence
        random_state=42
    )

    model = Pipeline([
        ('preproc', preproc),
        ('clf', clf)
    ])

    # Evaluate the model using cross-validation
    scores = cross_val_score(model, X, y, cv=StratifiedKFold(n_splits=5), n_jobs=-1, verbose=0)
    return scores.mean()

# Create an Optuna study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, n_jobs=-1, show_progress_bar=True)

print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Train the final model with the best parameters found by Optuna
best_mlp_clf = MLPClassifier(
    hidden_layer_sizes=trial.params['hidden_layer_sizes'],
    activation=trial.params['activation'],
    solver=trial.params['solver'],
    alpha=trial.params['alpha'],
    learning_rate_init=trial.params['learning_rate_init'],
    max_iter=1000,
    random_state=42
)

final_model = Pipeline([
    ('preproc', preproc),
    ('clf', best_mlp_clf)
])

final_model.fit(X, y)

# Make predictions on the test set with the best model
y_optuna_pred = final_model.predict(Xtest)

# Create submission file
submission_optuna = pd.read_csv('gender_submission.csv')
submission_optuna['Survived'] = y_optuna_pred
submission_optuna.to_csv('submission_optuna.csv', index=False)

print("Submission file 'submission_optuna.csv' created successfully.")

[I 2026-05-06 00:42:42,839] A new study created in memory with name: no-name-b9b83528-5650-4d9d-8651-ded5abd4fe74


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-06 00:42:56,518] Trial 1 finished with value: 0.8126357416358043 and parameters: {'hidden_layer_sizes': (100,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0021052867666084, 'learning_rate_init': 0.025486003049476923}. Best is trial 1 with value: 0.8126357416358043.
[I 2026-05-06 00:43:00,187] Trial 0 finished with value: 0.8069989328981231 and parameters: {'hidden_layer_sizes': (50, 50), 'activation': 'relu', 'solver': 'sgd', 'alpha': 0.00435106955706256, 'learning_rate_init': 0.012084193668537682}. Best is trial 1 with value: 0.8126357416358043.
[I 2026-05-06 00:43:02,926] Trial 2 finished with value: 0.7935220639005712 and parameters: {'hidden_layer_sizes': (50, 50), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0005742833078845184, 'learning_rate_init': 0.004029140312027307}. Best is trial 1 with value: 0.8126357416358043.
[I 2026-05-06 00:43:16,640] Trial 4 finished with value: 0.7969116816270165 and parameters: {'hidden_layer_sizes': (50, 50), 'activation':